In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="neutral_penalized_margin_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = model.config.id2label
label2id = {str(v).lower(): int(k) for k, v in id2label.items()}

def resolve_label_id(label2id_map, candidates):
    for cand in candidates:
        if cand in label2id_map:
            return label2id_map[cand]
    raise ValueError(f"Could not resolve label id from candidates={candidates}; available labels={label2id_map}")

contradiction_id = resolve_label_id(label2id, ["contradiction", "contradictory", "label_0"])
neutral_id = resolve_label_id(label2id, ["neutral", "label_1"])
entailment_id = resolve_label_id(label2id, ["entailment", "label_2"])

print(model_name)
print("id2label:", id2label)
print({
    "contradiction_id": contradiction_id,
    "neutral_id": neutral_id,
    "entailment_id": entailment_id,
})


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
id2label: {0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
{'contradiction_id': 2, 'neutral_id': 1, 'entailment_id': 0}


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 64
score_12_all = []
score_21_all = []
avg_score_all = []
preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc12 = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc12 = {k: v.to(device) for k, v in enc12.items()}

        enc21 = tokenizer(
            batch_s2,
            batch_s1,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc21 = {k: v.to(device) for k, v in enc21.items()}

        logits12 = model(**enc12).logits
        logits21 = model(**enc21).logits

        score12 = logits12[:, entailment_id] - torch.maximum(logits12[:, contradiction_id], logits12[:, neutral_id])
        score21 = logits21[:, entailment_id] - torch.maximum(logits21[:, contradiction_id], logits21[:, neutral_id])
        avg_score = (score12 + score21) / 2.0
        batch_preds = (avg_score > 0).long()

        score_12_all.extend(score12.cpu().numpy())
        score_21_all.extend(score21.cpu().numpy())
        avg_score_all.extend(avg_score.cpu().numpy())
        preds.extend(batch_preds.cpu().numpy())

score_12_all = np.array(score_12_all)
score_21_all = np.array(score_21_all)
avg_score_all = np.array(avg_score_all)
y_pred = np.array(preds)

print("done")
print("score_12 range:", float(score_12_all.min()), float(score_12_all.max()))
print("score_21 range:", float(score_21_all.min()), float(score_21_all.max()))
print("avg_score range:", float(avg_score_all.min()), float(avg_score_all.max()))


  0%|          | 0/7 [00:00<?, ?it/s]

done
score_12 range: -9.9390287399292 7.206862926483154
score_21 range: -10.324073791503906 7.335885524749756
avg_score range: -10.007959365844727 7.197781562805176


In [ ]:
vault.create_record_list("mrpc_neutral_penalty_prediction", column_names=["prediction", "score_12", "score_21"])

for i in range(len(y_pred)):
    vault.append_record("mrpc_neutral_penalty_prediction", 
                        {
                            "prediction": y_pred[i],
                            "score_12": float(score_12_all[i]),
                            "score_21": float(score_21_all[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT mrpc_neutral_penalty_prediction"
embedding = get_embeddings(description)
vault.create_description("mrpc_neutral_penalty_prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_neutral_penalty_prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.5563725490196079, 'f1': 0.5322997416020672}
                precision    recall  f1-score   support

not_paraphrase       0.41      0.96      0.58       129
    paraphrase       0.95      0.37      0.53       279

      accuracy                           0.56       408
     macro avg       0.68      0.67      0.56       408
  weighted avg       0.78      0.56      0.55       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score_12:", float(score_12_all[i]))
    print("score_21:", float(score_21_all[i]))
    print("avg_score:", float(avg_score_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", "paraphrase" if int(y_pred[i]) == 1 else "not_paraphrase")


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
score_12: 5.820394039154053
score_21: 1.4785807132720947
avg_score: 3.6494874954223633
true: 1 pred: 1 label: paraphrase
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
score_12: -9.776514053344727
score_21: -7.540008544921875
avg_score: -8.6582612991333
true: 0 pred: 0 label: not_paraphrase
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
score_12: -8.518594741821289
sc

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("score_12:", float(score_12_all[i]))
    print("score_21:", float(score_21_all[i]))
    print("avg_score:", float(avg_score_all[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


num_errors: 181
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
score_12: -9.537737846374512
score_21: 4.726302623748779
avg_score: -2.405717611312866
true: 1 pred: 0
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
score_12: -9.673195838928223
score_21: -8.567197799682617
avg_score: -9.120197296142578
true: 1 pred: 0
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C

In [8]:
vault.create_record_list("neutral_penalized_margin_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("neutral_penalized_margin_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_neutral_penalty_prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT neutral_penalized_margin_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("neutral_penalized_margin_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("neutral_penalized_margin_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.5563725490196079,
 'f1': 0.5322997416020672,
 'method': 'bidirectional_direct_nli_neutral_penalized_margin'}

In [ ]:
description = "INSERT TEXT HERE ABOUT neutral_penalized_margin_mrpc process/notebook" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("neutral_penalized_margin_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("neutral_penalized_margin_mrpc", cat, embedding, prop)